# 📊 Analyse Exploratoire des Données (EDA) : SuperMust/irac-thinking

Ce notebook permet d'analyser en profondeur le jeu de données juridique **`SuperMust/irac-thinking`** avant de l'envoyer au fine-tuning. L'objectif est d'évaluer sa qualité, de comprendre sa structure CoT (Chain-of-Thought) et de vérifier s'il est bien adapté pour le droit canadien/québécois.

## 🛠️ Étape 1 : Importation des packages et chargement du dataset

Nous commençons par charger les bibliothèques d'analyse et de visualisation classiques (`pandas`, `matplotlib`, `seaborn`, `collections`).

In [ ]:
# Installation des bibliothèques nécessaires si manquantes
!pip install datasets pandas matplotlib seaborn wordcloud jinja2

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset

# Configuration graphique
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

In [ ]:
# Chargement du dataset depuis Hugging Face
dataset_name = "SuperMust/irac-thinking"
print(f"Chargement du dataset {dataset_name}...")
dataset = load_dataset(dataset_name, split="train")
print(f"Dataset chargé ! Nombre d'exemples : {len(dataset)}")

## 🔍 Étape 2 : Analyse de la structure des exemples

Voyons comment les dialogues sont structurés. Nous allons parser le format conversationnel et extraire les rôles (`system`/`developer`, `user`, `assistant`), ainsi que les champs CoT (`thinking` et `content`).

In [ ]:
# Extraction structurée dans un DataFrame Pandas pour faciliter l'analyse
parsed_data = []

for idx, example in enumerate(dataset):
    messages = example.get("messages", [])
    
    # Initialisation des variables pour cet exemple
    system_prompt = ""
    user_prompt = ""
    assistant_thinking = ""
    assistant_content = ""
    
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        thinking = msg.get("thinking", "")
        
        if role in ["system", "developer"]:
            system_prompt = content
        elif role == "user":
            user_prompt = content
        elif role == "assistant":
            assistant_thinking = thinking
            assistant_content = content
            
    parsed_data.append({
        "id": idx,
        "system": system_prompt,
        "prompt": user_prompt,
        "thinking": assistant_thinking,
        "content": assistant_content,
        "has_thinking": bool(assistant_thinking),
        "prompt_len_chars": len(user_prompt),
        "prompt_words": len(user_prompt.split()),
        "thinking_len_chars": len(assistant_thinking),
        "thinking_words": len(assistant_thinking.split()) if assistant_thinking else 0,
        "content_len_chars": len(assistant_content),
        "content_words": len(assistant_content.split())
    })

df = pd.DataFrame(parsed_data)
print("Aperçu des premières lignes du DataFrame converti :")
df.head()

## 📈 Étape 3 : Statistiques descriptives globales

Analysons les distributions des longueurs de textes et la présence du raisonnement CoT.

In [ ]:
# Taux de présence du Chain-of-Thought
thinking_rate = df["has_thinking"].mean() * 100
print(f"Proportion d'exemples contenant une réflexion séparée ('thinking') : {thinking_rate:.2f}%")

# Description statistique des longueurs en mots
df[["prompt_words", "thinking_words", "content_words"]].describe()

In [ ]:
# Visualisation de la distribution de la longueur des mots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["prompt_words"], kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Distribution de la longueur des Prompts (mots)")
axes[0].set_xlabel("Nombre de mots")

sns.histplot(df[df["has_thinking"]]["thinking_words"], kde=True, ax=axes[1], color="orange")
axes[1].set_title("Distribution de la réflexion CoT (mots)")
axes[1].set_xlabel("Nombre de mots")

sns.histplot(df["content_words"], kde=True, ax=axes[2], color="green")
axes[2].set_title("Distribution de la Conclusion finale (mots)")
axes[2].set_xlabel("Nombre de mots")

plt.tight_layout()
plt.show()

## ⚖️ Étape 4 : Analyse de la Qualité du Raisonnement Juridique (Format IRAC)

Un bon dataset de distillation CoT juridique doit structurer son raisonnement selon la méthode **IRAC** (Issue, Rule, Application, Conclusion). Vérifions si ces concepts ou leurs variantes francophones apparaissent dans le bloc de réflexion.

In [ ]:
def check_irac_structure(text):
    if not text:
        return 0
    text_lower = text.lower()
    
    # Mots clés recherchés (anglais ou français)
    has_issue = any(w in text_lower for w in ["issue", "question", "problème"])
    has_rule = any(w in text_lower for w in ["rule", "règle", "loi", "législation", "article"])
    has_app = any(w in text_lower for w in ["application", "analyse", "faits"])
    has_concl = any(w in text_lower for w in ["conclusion", "résolution", "conclure"])
    
    score = sum([has_issue, has_rule, has_app, has_concl])
    return score

df["irac_score"] = df["thinking"].apply(check_irac_structure)

irac_counts = df["irac_score"].value_counts().sort_index()
print("Score de structure IRAC dans la réflexion CoT (sur 4) :")
for score, count in irac_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Score {score}/4 : {count} exemples ({percentage:.2f}%)")

## 🇨🇦 Étape 5 : Analyse de l'Alignement Droit Québécois / Canadien

Le modèle doit être entraîné sur la juridiction québécoise et canadienne. Vérifions la fréquence des termes géographiques et institutionnels clés.

In [ ]:
canadian_keywords = {
    "Québec/Quebec": re.compile(r"qu\b|qu\bebec|qu\b\u00e9bec", re.IGNORECASE),
    "Canada": re.compile(r"canada|canadien|canadienne", re.IGNORECASE),
    "Code civil (CCQ)": re.compile(r"ccq|code\s+civil\s+du\s+qu\b\u00e9bec|code\s+civil\s+qu\b\u00e9b\u00e9cois", re.IGNORECASE),
    "CanLII": re.compile(r"canlii", re.IGNORECASE),
    "Cour suprême": re.compile(r"cour\s+supr\u00eame|csc|scc", re.IGNORECASE),
    "Barreau / Éducaloi": re.compile(r"barreau|\beducaloi|\b\u00e9ducaloi", re.IGNORECASE)
}

keyword_stats = {}
for label, regex in canadian_keywords.items():
    # Recherche combinée dans le prompt, le thinking et le content
    matches = df.apply(lambda row: bool(regex.search(f"{row['prompt']} {row['thinking']} {row['content']}")), axis=1)
    keyword_stats[label] = matches.sum()

print("Présence de références canadiennes / québécoises dans le dataset :")
for label, count in keyword_stats.items():
    percentage = (count / len(df)) * 100
    print(f"  - {label} : {count} occurrences ({percentage:.2f}% des exemples)")

## 🛠️ Étape 6 : Analyse des formats JSON & Tool Calling (Spécificités Lexior)

LexiorNotebook s'appuie sur des formats d'appels d'outils et des citations au format JSON (`[^1]:{"type":"url",...}`). Voyons si ces structures sont déjà présentes et conformes.

In [ ]:
# 1. Présence d'appels d'outils (JSON ou inline)
has_tool_call_regex = re.compile(r"(search|get_case|browse_legislation|get_educaloi)\s*\{.*\}", re.IGNORECASE)
df["has_tool_call"] = df["content"].apply(lambda x: bool(has_tool_call_regex.search(x)) if x else False)

# 2. Présence de citations de bas de page au format JSON Lexior
has_footnote_json = re.compile(r"\[\^\d+\]:\s*\{\s*\"type\"\s*:\s*\"(url|doc|attachment)\"", re.IGNORECASE)
df["has_lexior_footnote"] = df["content"].apply(lambda x: bool(has_footnote_json.search(x)) if x else False)

print(f"Exemples avec appels d'outils (Tool Calling) : {df['has_tool_call'].sum()} ({df['has_tool_call'].mean()*100:.2f}%)")
print(f"Exemples avec citations formatées Lexior : {df['has_lexior_footnote'].sum()} ({df['has_lexior_footnote'].mean()*100:.2f}%)")

## ☁️ Étape 7 : Visualisation des mots-clés les plus fréquents

Générons un nuage de mots ou un graphique en barres des termes juridiques les plus représentatifs du jeu de données.

In [ ]:
# Récupération de tous les mots du raisonnement (thinking)
all_thinking_text = " ".join(df["thinking"].dropna().tolist()).lower()

# Nettoyage simple des stopwords communs en français
stopwords = set([
    "de", "la", "le", "et", "un", "une", "en", "que", "dans", "pour", "qui", 
    "des", "les", "du", "est", "au", "a", "par", "sur", "ce", "ces", "il", 
    "elle", "nous", "vous", "ils", "elles", "se", "sa", "son", "ses", "aux", 
    "pas", "ou", "d'un", "d'une", "l'un", "l'une", "qu'un", "qu'une", "avec"
])

words = re.findall(r"\b[a-z]{3,}\b", all_thinking_text)
filtered_words = [w for w in words if w not in stopwords]

word_freq = Counter(filtered_words)
common_words = word_freq.most_common(20)

# Affichage graphique sous forme de diagramme en barres
words_df = pd.DataFrame(common_words, columns=["Mot", "Fréquence"])
sns.barplot(data=words_df, x="Fréquence", y="Mot", palette="viridis")
plt.title("Top 20 des mots les plus fréquents dans la réflexion CoT (thinking)")
plt.show()

## 📋 Conclusion sur la qualité du dataset

Sur la base des analyses effectuées ci-dessus :

1. **Raisonnement CoT** : Vérifiez le pourcentage d'exemples contenant une réflexion (`has_thinking`). Un taux proche de 100% est nécessaire pour que le modèle local apprenne le comportement de raisonnement.
2. **Structure logique (IRAC)** : La présence des sections de qualification des faits, règles et application montre la rigueur juridique du modèle Teacher.
3. **Localisation Québec/Canada** : Si le taux de mots clés canadiens (CCQ, CanLII, Québec) est bas, le modèle risque d'halluciner des lois françaises ou américaines. Il est alors recommandé d'ajouter des paires de données spécifiques au droit canadien avant le fine-tuning.